In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-03 05:31:01.326815: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-03 05:31:02.162090: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": "local",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": "tensorflow",
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
  },
  "ML": {
      "algorithm": {
      "type": "KNN",
      "params": {
        "K": 5,
        "metric": "euclidean"
      }
    }    
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions


In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-03 05:31:03,492 [DEBUG] [Rain] Rain is initialized
2023-07-03 05:31:03,494 [DEBUG] [Provisioner] Creating coordinator


2023-07-03 05:31:03,495 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/data/
2023-07-03 05:31:03,495 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-03 05:31:03,496 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-03 05:31:03,497 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/data/
2023-07-03 05:31:03,498 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized
2023-07-03 05:31:03,499 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/
2023-07-03 05:31:03,500 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/
2023-07-03 05:31:03,501 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-03 05:31:03,506 [DEBUG] [Rain] Creating workers
2023-07-03 05:31:03,513 [INFO] [Provisioner] provisioner is serving
2023-07-03 05:31:03,514 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 05:31:03,516 [INFO] [Coordinator] coordinator is serving
2023-07-03 05:31:03,517 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 05:31:03,520 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 05:31:03,522 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 05:31:03,523 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 05:31:03,523 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 05:31:03,525 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 05:31:03,526 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 05:31:03,528

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 10ms/step - loss: 0.7101 - accuracy: 0.7757
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.7252 - accuracy: 0.7706
Epoch 2/2
157/157 [==============================] - 2s 13ms/step - loss: 0.3135 - accuracy: 0.9060
sending data to coordinator
sending data to coordinator


2023-07-03 05:31:45,422 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 05:31:45,427 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 05:31:45,439 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 05:31:45,441 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 05:31:45,490 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 05:31:45,492 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 05:31:45,693 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 05:31:45,712 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 3s 12ms/step - loss: 0.3027 - accuracy: 0.9105
Epoch 2/2
157/157 [==============================] - 3s 11ms/step - loss: 0.3537 - accuracy: 0.8959
Epoch 2/2
157/157 [==============================] - 2s 12ms/step - loss: 0.2288 - accuracy: 0.9340
sending data to coordinator
sending data to coordinator


2023-07-03 05:31:55,679 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 05:31:55,680 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 05:31:55,705 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 05:31:55,707 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 05:31:55,732 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 05:31:55,734 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 05:31:55,912 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2 successfully
2023-07-03 05:31:55,922 [DEBUG] [DeepLearning] Asynchronous update is done by

Epoch 1/2
Epoch 1/2
Epoch 1/2


157/157 [==============================] - 2s 10ms/step - loss: 0.2041 - accuracy: 0.9395
Epoch 2/2
157/157 [==============================] - 3s 10ms/step - loss: 0.2243 - accuracy: 0.9311
Epoch 2/2
157/157 [==============================] - 3s 9ms/step - loss: 0.1922 - accuracy: 0.9439
Epoch 2/2
157/157 [==============================] - 2s 11ms/step - loss: 0.1573 - accuracy: 0.9542
sending data to coordinator


2023-07-03 05:32:05,791 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 05:32:05,793 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 05:32:05,873 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 05:32:05,880 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
2023-07-03 05:32:05,902 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
2023-07-03 05:32:05,977 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 05:32:05,978 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 05:32:06,019 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 05:32:06,020 [DEBUG] [DividerAmbassado

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 6ms/step - loss: 0.1082 - accuracy: 0.9669

Test accuracy: 96.7%


In [11]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-03 05:33:13,441 [DEBUG] [Rain] Creating workers
2023-07-03 05:33:13,448 [INFO] [Provisioner] provisioner is serving
2023-07-03 05:33:13,450 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 05:33:13,451 [INFO] [Coordinator] coordinator is serving
2023-07-03 05:33:13,452 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 05:33:13,454 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 05:33:13,455 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 05:33:13,457 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 05:33:13,458 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 05:33:13,459 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 05:33:13,459 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 05:33:13,461 [DEBUG] [TemporaryFilesManager] Creating

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0766 - accuracy: 0.9776

Test accuracy: 97.8%
